# 04 — Simulazione "OASIS-inspired" (Blocco C)

**A cosa serve questo notebook.** È il "cruscotto" del Blocco C: qui **non** si fa il lavoro pesante (gli scenari coi modelli grandi girano su HPC), ma si
1. prova il meccanismo in **locale** con un modello finto (mock), senza HPC;
2. si **caricano** i risultati calcolati su HPC (`data/processed/simulazioni/<scenario>/<modello>/`);
3. si **confrontano i modelli** sullo stesso scenario (tabella di sintesi + mappe interattive affiancate) — materiale per la validazione manuale.

Ogni run produce: `risultato.json` (traccia completa: reazioni round-per-round, cambi di stato e di archi, snapshot del grafo) e `mappa.html` (mappa-simulazione interattiva).

In [ ]:
import sys
from pathlib import Path
RADICE = Path.cwd().parent
sys.path.insert(0, str(RADICE))

import pandas as pd
from IPython.display import IFrame, display
from src.simulation.scenari import SCENARI, elenco
from src.simulation.run_simulazione import carica, riepilogo, scenari_disponibili, modelli_disponibili

print("Scenari definiti:", elenco())

## 1) Prova locale (mock, senza HPC)
Reazioni **finte** e deterministiche: serve solo a vedere il meccanismo (propagazione, cambi di stato/archi, mappa). I modelli veri girano su HPC.

In [ ]:
from src.simulation.oasis_inspired import simula, responder_mock
from src.simulation.mappa_sim import genera

res = simula(SCENARI["siccita_darfur"], responder_mock, n_round=4, verbose=True)
genera(res, titolo="demo_locale",
       out_path=RADICE/"data/processed/graphs/simulazioni/demo_locale.html")
IFrame(src="../data/processed/graphs/simulazioni/demo_locale.html", width="100%", height=560)

## 2) Risultati calcolati su HPC
Dopo aver lanciato il job su Leonardo, i risultati stanno in `data/processed/simulazioni/`. Scegli scenario e modello e guarda la mappa.

In [ ]:
SCEN = "siccita_darfur"    # <-- scegli lo scenario
print("scenari disponibili:", scenari_disponibili())
print(f"modelli per '{SCEN}':", modelli_disponibili(SCEN))

MOD = (modelli_disponibili(SCEN) or ["mock"])[0]   # <-- scegli il modello
print("mostro:", SCEN, "/", MOD)
IFrame(src=f"../data/processed/simulazioni/{SCEN}/{MOD}/mappa.html", width="100%", height=560)

## 3) Confronto tra modelli
Sullo stesso scenario: una **tabella di sintesi** (round, Paesi che hanno reagito, archi creati/tagliati/rafforzati/indeboliti, cambi di stato) e le **mappe** dei vari modelli, per trovarne le differenze.

In [ ]:
SCEN = "siccita_darfur"    # <-- scenario da confrontare
mods = modelli_disponibili(SCEN)

righe = [{"modello": m, **riepilogo(carica(SCEN, m))} for m in mods if carica(SCEN, m)]
tab = pd.DataFrame(righe).set_index("modello") if righe else pd.DataFrame()
display(tab)

for m in mods:
    print(f"=== {SCEN} · {m} ===")
    display(IFrame(src=f"../data/processed/simulazioni/{SCEN}/{m}/mappa.html", width="100%", height=520))